<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook A06: Evaluating Models</h2>
</div>

Worked solutions to the 3 exercises in
[Notebook A06: Evaluating Models](../notebooks/A06_Evaluating_models.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The series, the split, the baselines and the five metrics from the notebook.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

HORIZON = 12
SEASON_LENGTH = 12
TEST_MONTHS = 24

train, test = series.iloc[:-TEST_MONTHS], series.iloc[-TEST_MONTHS:]

last_cycle = train.iloc[-SEASON_LENGTH:].to_numpy()
seasonal_naive = pd.Series(
    [last_cycle[i % SEASON_LENGTH] for i in range(TEST_MONTHS)], index=test.index
)

SCALE = np.mean(np.abs(train.values[SEASON_LENGTH:] - train.values[:-SEASON_LENGTH]))


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


def root_mean_squared_error(actual, forecast):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def forecast_bias(actual, forecast):
    return float(np.mean(np.asarray(forecast) - np.asarray(actual)))


def mean_absolute_percentage_error(actual, forecast):
    actual, forecast = np.asarray(actual), np.asarray(forecast)
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


def mean_absolute_scaled_error(actual, forecast):
    return mean_absolute_error(actual, forecast) / SCALE


def all_metrics(actual, forecast):
    return {
        "MAE": mean_absolute_error(actual, forecast),
        "RMSE": root_mean_squared_error(actual, forecast),
        "Bias": forecast_bias(actual, forecast),
        "MAPE": mean_absolute_percentage_error(actual, forecast),
        "MASE": mean_absolute_scaled_error(actual, forecast),
    }


print(f"Train {len(train)} months, test {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Compute all five metrics for a forecast that always predicts the training median. Where does it rank? Then construct a deliberately biased forecast by adding 3 °C to the seasonal naive values, and check which metrics notice and which do not.

In [ ]:
median_forecast = pd.Series(train.median(), index=test.index)
biased_forecast = seasonal_naive + 3.0

candidates = {
    "Seasonal naive": seasonal_naive,
    "Training median": median_forecast,
    "Seasonal naive + 3 °C": biased_forecast,
}

comparison = pd.DataFrame(
    {name: all_metrics(test, forecast) for name, forecast in candidates.items()}
).T

print(f"Training median: {train.median():.2f} °C   (training mean: {train.mean():.2f} °C)\n")
comparison.round(2)

**The training median ranks with the other flat forecasts**, at 6.08 MAE. That is essentially the Mean
baseline's 6.04 from the notebook, which is no surprise: the median (8.63 °C) and the mean (8.86 °C) are
close on a series this symmetric, so the two forecasts are nearly the same constant. The median would pull
away from the mean on a skewed series, and on this one it has nothing to do.

The biased forecast is the more interesting half, and the answer is not "some metrics notice and some do
not". **Every metric gets worse.** MAE goes 1.74 to 2.52, RMSE 2.17 to 3.03, MAPE 36 to 66%, MASE 0.85 to
1.24. Adding a constant error to a good forecast degrades it, and all five measure that.

What separates them is *what they tell you about the degradation*:

**Only Bias identifies it as a systematic offset.** It moves from -0.75 to +2.25, and the difference is
exactly +3.00 — the size of the bias we injected. No other metric can distinguish "the forecast is three
degrees too high, every single month" from "the forecast has become noisier by a similar amount". Those
call for completely different fixes: the first is one subtraction away from being corrected, the second is
a modelling problem.

Note also that MASE crosses 1.0, from 0.85 to 1.24. That threshold is the whole point of the metric: the
biased forecast is now *worse* than the naive benchmark it is scaled against, and MASE says so in a way
that reading "2.52 °C" does not.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Re-run the rolling evaluation with `horizon=1` instead of 12. Do the errors get smaller, and does the gap between the three methods change? What does that tell you about comparing forecasts made at different horizons?

In [ ]:
def rolling_origin(series, horizon, n_origins=24, step=12):
    """Mean MAE across several origins, for the three baselines."""
    errors = {"Seasonal naive": [], "Mean": [], "Naive": []}

    for k in range(n_origins):
        end = len(series) - horizon - (n_origins - 1 - k) * step
        history, actual = series.iloc[:end], series.iloc[end:end + horizon]

        cycle = history.iloc[-SEASON_LENGTH:].to_numpy()
        errors["Seasonal naive"].append(
            mean_absolute_error(actual, [cycle[i % SEASON_LENGTH] for i in range(horizon)])
        )
        errors["Mean"].append(mean_absolute_error(actual, [history.mean()] * horizon))
        errors["Naive"].append(mean_absolute_error(actual, [history.iloc[-1]] * horizon))

    return {name: float(np.mean(values)) for name, values in errors.items()}


pd.DataFrame({
    "horizon 1": rolling_origin(series, horizon=1),
    "horizon 12": rolling_origin(series, horizon=12),
}).round(2)

This table is strange, and the strangeness is the exercise.

Two of the three behave as expected. **Seasonal naive** improves slightly at the shorter horizon, 1.53
against 2.00. **Naive** improves enormously, 1.56 against 9.35, which makes sense: predicting one month
ahead with last month's value is reasonable, and predicting twelve months ahead with it is hopeless.

But **the Mean gets worse at the shorter horizon**, 10.27 against 6.08. A flat forecast should not care
about the horizon at all, so something is wrong with the comparison rather than with the method.

In [ ]:
targets = []
for k in range(24):
    end = len(series) - 1 - (23 - k) * 12
    targets.append(series.index[end])

print("Months targeted when horizon=1 and step=12:")
print(" ", sorted({timestamp.strftime("%B") for timestamp in targets}))
print(f"\nAugust averages {series[series.index.month == 8].mean():.2f} °C, "
      f"the series as a whole {series.mean():.2f} °C")

**Every one of the 24 origins forecasts August.** With a step of 12 months and a horizon of 1, each
origin lands exactly a year after the last, so the evaluation is not measuring performance on the series
— it is measuring performance on one month of the year.

That explains the Mean's 10.27 exactly. August averages 17.69 °C against the series mean of 8.89, so a flat
forecast at the training mean is wrong by about 8.8 °C at every single origin. It is not a bad forecast of
the series; it is being asked, twenty-four times, the one question it answers worst.

The fix is to space the origins so that all months are represented.

In [ ]:
pd.DataFrame({
    "horizon 1": rolling_origin(series, horizon=1, step=1),
    "horizon 12": rolling_origin(series, horizon=12, step=1),
}).round(2)

With overlapping origins the picture is sane. **The Mean now scores 6.04 at both horizons**, which is what
a flat forecast must do: its error cannot depend on how far ahead you ask. Seasonal naive barely moves
either, 1.81 against 1.62, which is right for a method that uses a year-old observation whichever horizon
you ask about. Only Naive shows a large horizon effect, 3.15 against 7.38, because it is the only one of
the three whose information decays as you forecast further out.

Three things to take from this:

**Errors do generally shrink at shorter horizons**, and the effect is largest for methods that lean on the
most recent observation. Naive loses a factor of six between horizon 12 and horizon 1; seasonal naive
barely moves, because what it leans on is a year old either way.

**Comparing errors across horizons is meaningless without saying so.** A model reported at 1.5 °C for
one-step-ahead and one reported at 2.0 °C for a year ahead may well be the same model.

**And the evaluation design can dominate the result.** Nothing about the first table was a bug: the code
did what it was told. But the combination of `horizon=1` and `step=12` quietly restricted the test to a
single calendar month, and one of the three methods was crippled by that while the others were not. Before
believing a comparison, check what it is actually averaging over.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Fit the Holt-Winters model from Notebook A05 (`ExponentialSmoothing` with additive trend and season) and run the same two diagnostics on `fitted.resid`. Is its ACF cleaner than the seasonal naive one? Does Ljung-Box still reject independence?

In [ ]:
holt_winters = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=SEASON_LENGTH
).fit()

seasonal_naive_residuals = (series - series.shift(SEASON_LENGTH)).dropna()
holt_winters_residuals = holt_winters.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (name, residuals) in zip(
    axes,
    [("Seasonal naive", seasonal_naive_residuals), ("Holt-Winters", holt_winters_residuals)],
):
    plot_acf(residuals, lags=36, ax=ax, color="steelblue", vlines_kwargs={"colors": "steelblue"})
    ax.set_title(f"{name} residuals", fontsize=13, fontweight="bold")
    ax.set_xlabel("Lag (months)")
    ax.set_ylim(-0.7, 0.7)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import acf

summary = []
for name, residuals in [("Seasonal naive", seasonal_naive_residuals),
                        ("Holt-Winters", holt_winters_residuals)]:
    autocorrelation = acf(residuals, nlags=24)
    ljung_box = acorr_ljungbox(residuals, lags=[12, 24], return_df=True)
    summary.append({
        "model": name,
        "residual std": residuals.std(),
        "ACF lag 1": autocorrelation[1],
        "ACF lag 12": autocorrelation[12],
        "Ljung-Box p (12)": ljung_box["lb_pvalue"].iloc[0],
        "Ljung-Box p (24)": ljung_box["lb_pvalue"].iloc[1],
    })

pd.DataFrame(summary).set_index("model")

**Yes, the ACF is much cleaner — and no, Ljung-Box still rejects.** Both halves matter.

The improvement is real and specific. The seasonal naive residuals carry a correlation of **-0.50 at lag
12**; Holt-Winters reduces that to **-0.02**. The seasonal structure that the naive method left behind has
been almost entirely absorbed, which is exactly what a model with a seasonal component is for. The residual
standard deviation falls from 2.69 to 1.92 at the same time.

But **the lag-1 correlation is untouched**: +0.209 for the naive residuals, +0.201 for Holt-Winters. The
model removed the seasonality and left the short-range persistence completely alone, and it is that
persistence which keeps Ljung-Box rejecting, at p ≈ 1e-12.

The p-value is worth reading carefully rather than as a verdict. It fell from about 1e-109 to about 1e-12,
which is an enormous improvement and still overwhelming rejection. With 1,700 observations, a correlation
of 0.2 is detected with certainty. **A significant Ljung-Box result on a long series tells you structure
remains, not that the model is bad**, and the useful question is what kind of structure and whether you
can model it.

Here the answer is visible: one dominant spike at lag 1, and little else. That is the signature of an
autoregressive term, and adding one is precisely what Notebook
[B02](../notebooks/B02_ARIMA_models.ipynb) does. Its searched model lands on an AR component plus a
seasonal MA term — the two things these residual plots have been pointing at.

---

Back to [Notebook A06](../notebooks/A06_Evaluating_models.ipynb), or on to
[Notebook B01](../notebooks/B01_Exponential_smoothing_models.ipynb).